In [2]:
import requests
import glob
import pandas as pd
import os
import yfinance as yf
import numpy as np
import time
from datetime import datetime as dt, timedelta
from bs4 import BeautifulSoup
import pytz

In [7]:
###     NEWS SCRAPER Z MARKETINDSIDER (gets rid-off seekingalpha, beware look-ahead bias)
def is_premarket_or_aftermarket(article_time, target_date):
    market_open = dt(target_date.year, target_date.month, target_date.day, 15, 30)   
    market_close = dt(target_date.year, target_date.month, target_date.day, 22, 0)  
    previous_close = market_close - timedelta(days=1)
    
    return previous_close < article_time < market_open

def scrape_news(stock_symbol):
    columns = ['datetime', 'title', 'source', 'link']
    df = pd.DataFrame(columns=columns)
    counter = 0
    
    for page in range(1, 3):
        url = f'https://markets.businessinsider.com/news/{stock_symbol}-stock?p={page}'
        headers = {'User-Agent': 'Mozilla/5.0'}
        response = requests.get(url, headers=headers)
        html = response.text
        soup = BeautifulSoup(html, 'lxml')
        articles = soup.find_all('div', class_ = 'latest-news__story')
        
        for article in articles:
            datetime_str = article.find('time', class_ = 'latest-news__date').get('datetime')
            title = article.find('a', class_ = 'news-link').text
            source = article.find('span', class_ = 'latest-news__source').text
            link = article.find('a', class_ = 'news-link').get('href')
            
            try:
                news_datetime = dt.strptime(datetime_str, '%m/%d/%Y %I:%M:%S %p')
            except ValueError as e:
                print(f"Error parsing datetime: {datetime_str}, {e}")
                continue

            df = pd.concat([pd.DataFrame([[news_datetime, title, source, link]], columns=df.columns), df], ignore_index=True)
            counter += 1
            
        time.sleep(0.2)
        
    print(f'{counter} articles scraped')
    return df

# GAP DAY
target_date = dt(2024, 11, 8)
tick_news = scrape_news('aaoi')

tick_news['datetime'] = pd.to_datetime(tick_news['datetime'])
filtered_news = tick_news[tick_news['datetime'].apply(lambda x: is_premarket_or_aftermarket(x, target_date))]
filtered_news = filtered_news[filtered_news['source'] != 'Seeking Alpha']
filtered_news


100 articles scraped


,datetime,title,source,link
93,2024-11-07 22:05:00,Applied Optoelectronics Reports Third Quarter ...,GlobeNewswire,/news/stocks/applied-optoelectronics-reports-t...
94,2024-11-07 22:12:57,Applied Optoelectronics Enters New Equity Dist...,TipRanks,/news/stocks/applied-optoelectronics-enters-ne...
95,2024-11-08 00:40:57,Closing Bell Movers: Toast gains 19% afterhour...,TipRanks,/news/stocks/closing-bell-movers-toast-gains-1...
96,2024-11-08 05:19:14,Applied Optoelectronics Reports Q3 2024 Financ...,TipRanks,/news/stocks/applied-optoelectronics-reports-q...
97,2024-11-08 13:16:00,Applied Optoelectronics price target raised to...,TipRanks,/news/stocks/applied-optoelectronics-price-tar...
98,2024-11-08 13:46:44,Applied Optoelectronics price target raised to...,TipRanks,/news/stocks/applied-optoelectronics-price-tar...
99,2024-11-08 14:40:10,Analysts Offer Insights on Technology Companie...,TipRanks,/news/stocks/analysts-offer-insights-on-techno...


In [8]:
# Function to check if an article is premarket or aftermarket based on user-defined times
def is_premarket_or_aftermarket(article_time, target_date):
    market_open = dt(target_date.year, target_date.month, target_date.day, 15, 30)
    market_close = dt(target_date.year, target_date.month, target_date.day, 22, 0)
    previous_close = market_close - timedelta(days=1)
    
    return previous_close < article_time < market_open

# Function to scrape the news articles (datetime, title, source, link)
def scrape_news(stock_symbol):
    columns = ['datetime', 'title', 'source', 'link']
    df = pd.DataFrame(columns=columns)
    counter = 0
    
    for page in range(1, 5):
        url = f'https://markets.businessinsider.com/news/{stock_symbol}-stock?p={page}'
        headers = {'User-Agent': 'Mozilla/5.0'}
        response = requests.get(url, headers=headers)
        html = response.text
        soup = BeautifulSoup(html, 'lxml')
        articles = soup.find_all('div', class_ = 'latest-news__story')
        
        for article in articles:
            datetime_str = article.find('time', class_ = 'latest-news__date').get('datetime')
            title = article.find('a', class_ = 'news-link').text
            source = article.find('span', class_ = 'latest-news__source').text
            link = article.find('a', class_ = 'news-link').get('href')
            
            try:
                news_datetime = dt.strptime(datetime_str, '%m/%d/%Y %I:%M:%S %p')
            except ValueError as e:
                print(f"Error parsing datetime: {datetime_str}, {e}")
                continue

            df = pd.concat([pd.DataFrame([[news_datetime, title, source, link]], columns=df.columns), df], ignore_index=True)
            counter += 1
            
        time.sleep(0.2)
        
    print(f'{counter} articles scraped')
    return df

# Function to scrape full content of articles using the links
def scrape_content(link):
    base_url = 'https://markets.businessinsider.com'
    full_url = base_url + link
    headers = {'User-Agent': 'Mozilla/5.0'}
    
    response = requests.get(full_url, headers=headers)
    if response.status_code == 200:
        soup = BeautifulSoup(response.text, 'lxml')
        # Assuming the article text is within paragraphs <p>
        article_text = ' '.join([p.get_text() for p in soup.find_all('p')])
        return article_text
    else:
        return None

# Scrape news and filter based on premarket or aftermarket news
target_date = dt(2024, 11, 8)
stock_news = scrape_news('aaoi')
stock_news['datetime'] = pd.to_datetime(stock_news['datetime'])

# Filter news articles for premarket on the target date
filtered_news = stock_news[stock_news['datetime'].apply(lambda x: is_premarket_or_aftermarket(x, target_date))]
filtered_news = filtered_news[filtered_news['source'] != 'Seeking Alpha']

# Scrape full content for the filtered news articles
filtered_news['content'] = filtered_news['link'].apply(scrape_content)
filtered_news



200 articles scraped


,datetime,title,source,link,content
193,2024-11-07 22:05:00,Applied Optoelectronics Reports Third Quarter ...,GlobeNewswire,/news/stocks/applied-optoelectronics-reports-t...,"SUGAR LAND, Texas, Nov. 07, 2024 (GLOBE NEWS..."
194,2024-11-07 22:12:57,Applied Optoelectronics Enters New Equity Dist...,TipRanks,/news/stocks/applied-optoelectronics-enters-ne...,Applied Optoelectronics ( (AAOI) ) has issued...
195,2024-11-08 00:40:57,Closing Bell Movers: Toast gains 19% afterhour...,TipRanks,/news/stocks/closing-bell-movers-toast-gains-1...,Check out this evening’s top movers from aroun...
196,2024-11-08 05:19:14,Applied Optoelectronics Reports Q3 2024 Financ...,TipRanks,/news/stocks/applied-optoelectronics-reports-q...,Applied Optoelectronics Inc ( (AAOI) ) has rel...
197,2024-11-08 13:16:00,Applied Optoelectronics price target raised to...,TipRanks,/news/stocks/applied-optoelectronics-price-tar...,Rosenblatt analyst Mike Genovese raised the fi...
198,2024-11-08 13:46:44,Applied Optoelectronics price target raised to...,TipRanks,/news/stocks/applied-optoelectronics-price-tar...,B. Riley raised the firm’s price target on App...
199,2024-11-08 14:40:10,Analysts Offer Insights on Technology Companie...,TipRanks,/news/stocks/analysts-offer-insights-on-techno...,There’s a lot to be optimistic about in the Te...


In [9]:
def save_news_to_txt(filtered_news, file_path):
    with open(file_path, 'w', encoding='utf-8') as file:
        for index, row in filtered_news.iterrows():
            file.write(f"Title: {row['title']}\n")
            file.write(f"Source: {row['source']}\n")
            file.write(f"Datetime: {row['datetime']}\n")
            file.write(f"Link: {row['link']}\n")
            file.write(f"Content: {row['content']}\n")  # Assuming 'content' column is filled with full article text
            file.write("\n" + "="*80 + "\n\n")  # Separate articles with lines for better readability
    print(f"News saved to {file_path}")

# call function
save_news_to_txt(filtered_news, 'C:/Users/j-vas/1JUPYTERLAB/diplomka/filtered_news4.txt')


News saved to C:/Users/j-vas/1JUPYTERLAB/diplomka/filtered_news4.txt
